# AutoGluon — Train & Export for Inference Agent

Trains AutoGluon TabularPredictor for 4 diseases and exports model artifacts
to S3 for the Model Inference Agent.

**Two training modes:**
- `local` — Train directly with TabularPredictor (any machine)
- `jumpstart` — Use SageMaker JumpStart `autogluon-classification-ensemble` (SageMaker Studio)

**Diseases:** Hypertension, Stroke, CHD, Alzheimer's
**Dataset:** Synthea Coherent (151 patients, combined clinical + genomic + imaging)

## 0. Setup

In [ ]:
import sys, time, json, shutil, logging, warnings
import pandas as pd
import numpy as np
import boto3
from pathlib import Path
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

In [ ]:
# ═══ CONFIGURATION ═══
AUTOGLUON_MODE = "local"  # "local" or "jumpstart"
SAGEMAKER_TRAINING_INSTANCE = "ml.p3.2xlarge"
SAGEMAKER_INFERENCE_INSTANCE = "ml.m5.2xlarge"

S3_BUCKET = "guidance-multimodal-hcls-healthai-machinelearning"
S3_MODEL_PREFIX = "models"

USE_FEATURE_STORE = False  # True = SageMaker FeatureStore, False = local CSVs
LOCAL_DATA_DIR = "../data/raw/preprocessing"  # Adjust path as needed
LOCAL_EXPORT_DIR = Path("../models/exported")
LOCAL_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
UPLOAD_TO_S3 = False

DISEASE_CODES = {
    'alzheimers_prediction': '26929004',
    'coronary_heart_disease_prediction': '53741008',
    'stroke_prediction': '230690007',
    'hypertension_prediction': '59621000',
}
DISEASES = list(DISEASE_CODES.keys())

OPTIMAL_SEEDS = {
    'alzheimers_prediction': 36,
    'coronary_heart_disease_prediction': 2,
    'stroke_prediction': 30,
    'hypertension_prediction': 11,
}

BENCHMARK_F1 = {
    "hypertension_prediction": 0.909,
    "stroke_prediction": 1.000,
    "coronary_heart_disease_prediction": 0.667,
    "alzheimers_prediction": 0.400,
}

print(f"Mode: {AUTOGLUON_MODE}")
print(f"Diseases: {DISEASES}")

## 1. Load Data

In [ ]:
# Path A: SageMaker FeatureStore
if USE_FEATURE_STORE:
    import sagemaker
    from sagemaker.session import Session
    from sagemaker import get_execution_role
    from sagemaker.feature_store.feature_group import FeatureGroup

    sm_session = sagemaker.Session()
    region = boto3.Session().region_name
    role = get_execution_role()
    default_bucket = sm_session.default_bucket()

    boto_session = boto3.Session(region_name=region)
    feature_store_session = Session(
        boto_session=boto_session,
        sagemaker_client=boto_session.client('sagemaker', region_name=region),
        sagemaker_featurestore_runtime_client=boto_session.client('sagemaker-featurestore-runtime', region_name=region),
    )

    genomic_fg = FeatureGroup(name='genomic-feature-group', sagemaker_session=feature_store_session)
    clinical_fg = FeatureGroup(name='clinical-feature-group', sagemaker_session=feature_store_session)
    imaging_fg = FeatureGroup(name='imaging-feature-group', sagemaker_session=feature_store_session)

    PatientID = 'patientid'
    gq, cq, iq = genomic_fg.athena_query(), clinical_fg.athena_query(), imaging_fg.athena_query()

    query_string = f'''SELECT * FROM "{gq.table_name}", "{cq.table_name}", "{iq.table_name}"
                       WHERE "{gq.table_name}".{PatientID} = "{cq.table_name}".{PatientID}
                       AND "{gq.table_name}".{PatientID} = "{iq.table_name}".{PatientID}
                       ORDER BY "{cq.table_name}".{PatientID} ASC'''

    output_loc = f's3://{default_bucket}/multi-model-health-ml/feature-store-queries'
    gq.run(query_string=query_string, output_location=output_loc)
    gq.wait()
    combined = gq.as_dataframe()
    combined = combined.astype(str).replace({"{": "", "}": ""}, regex=True)

# Path B: Local CSVs
else:
    sys.path.insert(0, '..')
    from src.data.preprocessing_v2 import load_and_preprocess_all, build_combined_dataset
    modalities = load_and_preprocess_all(LOCAL_DATA_DIR)
    combined = build_combined_dataset(modalities)

print(f"Dataset: {combined.shape[0]} patients, {combined.shape[1]} features")

## 2. Train/Test Split

In [ ]:
def get_split(dataset, target, seed):
    other_targets = [d for d in DISEASES if d != target]
    drop_cols = ['patientID'] + other_targets
    drop_cols = [c for c in drop_cols if c in dataset.columns]
    train_df, test_df = train_test_split(dataset, test_size=0.2, random_state=seed, stratify=dataset[target])
    return train_df.drop(columns=drop_cols), test_df.drop(columns=drop_cols)

for d in DISEASES:
    tr, te = get_split(combined, d, OPTIMAL_SEEDS[d])
    print(f"{d}: train={len(tr)}, test={len(te)}, test_pos={(te[d]=='1').sum()}")

## 3. Train AutoGluon
Mode A (local): TabularPredictor directly
Mode B (jumpstart): SageMaker JumpStart Estimator

In [ ]:
from autogluon.tabular import TabularPredictor

def train_local(train, test, target, save_path):
    X_test = test.drop(columns=[target])
    start = time.time()
    predictor = TabularPredictor(
        label=target, problem_type='binary', path=save_path, verbosity=0,
    ).fit(train_data=train, holdout_frac=0.1, excluded_model_types=['CAT', 'XGB'])
    train_time = time.time() - start

    y_pred = predictor.predict(X_test)
    y_true = test[target]
    return predictor, {
        'f1': round(f1_score(y_true, y_pred, pos_label='1', zero_division=0), 4),
        'accuracy': round(accuracy_score(y_true, y_pred), 4),
        'precision': round(precision_score(y_true, y_pred, pos_label='1', zero_division=0), 4),
        'recall': round(recall_score(y_true, y_pred, pos_label='1', zero_division=0), 4),
        'training_time_s': round(train_time, 2),
    }


def train_jumpstart(train, test, target, disease_name):
    """SageMaker JumpStart training — matches original guidance repo."""
    from sagemaker import image_uris, model_uris, script_uris, hyperparameters
    from sagemaker.estimator import Estimator
    from sagemaker.utils import name_from_base

    role = get_execution_role()
    model_id, model_ver = "autogluon-classification-ensemble", "*"

    # Prepare SageMaker format (Target as first column)
    train_sm = train.copy()
    target_col = train_sm.pop(target)
    train_sm.insert(0, 'Target', target_col)
    train_sm.columns = ["Target"] + [f"Feature_{i}" for i in range(1, train_sm.shape[1])]

    train_s3_key = f"multi-model-health-ml/{disease_name}/train/data.csv"
    train_sm.to_csv(f"/tmp/{disease_name}_train.csv", index=False)
    s3_client = boto3.client('s3')
    s3_client.upload_file(f"/tmp/{disease_name}_train.csv", default_bucket, train_s3_key)

    train_image = image_uris.retrieve(region=None, framework=None, model_id=model_id,
        model_version=model_ver, image_scope="training", instance_type=SAGEMAKER_TRAINING_INSTANCE)
    train_source = script_uris.retrieve(model_id=model_id, model_version=model_ver, script_scope="training")
    train_model = model_uris.retrieve(model_id=model_id, model_version=model_ver, model_scope="training")

    hp = hyperparameters.retrieve_default(model_id=model_id, model_version=model_ver)
    hp["auto_stack"] = "True"

    estimator = Estimator(
        role=role, image_uri=train_image, source_dir=train_source, model_uri=train_model,
        entry_point="transfer_learning.py", instance_count=1, instance_type=SAGEMAKER_TRAINING_INSTANCE,
        max_run=360000, hyperparameters=hp,
        output_path=f"s3://{default_bucket}/multi-model-health-ml/{disease_name}/output",
    )

    start = time.time()
    estimator.fit({"training": f"s3://{default_bucket}/{train_s3_key}"}, logs=True,
                  job_name=name_from_base(f"ag-{disease_name[:20]}"))
    train_time = time.time() - start

    # Deploy, evaluate, cleanup
    deploy_image = image_uris.retrieve(region=None, framework=None, image_scope="inference",
        model_id=model_id, model_version=model_ver, instance_type=SAGEMAKER_INFERENCE_INSTANCE)
    deploy_source = script_uris.retrieve(model_id=model_id, model_version=model_ver, script_scope="inference")
    ep_name = name_from_base(f"ag-{disease_name[:15]}")
    predictor_ep = estimator.deploy(initial_instance_count=1, instance_type=SAGEMAKER_INFERENCE_INSTANCE,
        entry_point="inference.py", image_uri=deploy_image, source_dir=deploy_source, endpoint_name=ep_name)

    test_sm = test.copy()
    test_target = test_sm.pop(target)
    test_sm.columns = [f"Feature_{i}" for i in range(1, test_sm.shape[1] + 1)]

    import json as _json
    resp = boto3.client("runtime.sagemaker").invoke_endpoint(
        EndpointName=ep_name, ContentType="text/csv",
        Body=test_sm.to_csv(header=False, index=False).encode("utf-8"))
    preds = _json.loads(resp["Body"].read())
    y_pred = np.argmax(np.array(preds["probabilities"]), axis=1)
    y_true = test_target.astype(int).values

    predictor_ep.delete_model()
    predictor_ep.delete_endpoint()

    return estimator, {
        'f1': round(f1_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
        'accuracy': round(accuracy_score(y_true, y_pred), 4),
        'training_time_s': round(train_time, 2),
        'model_data': estimator.model_data,
    }

In [ ]:
results = {}
for disease in DISEASES:
    seed = OPTIMAL_SEEDS[disease]
    train, test = get_split(combined, disease, seed)
    disease_short = disease.replace('_prediction', '')

    print(f"\n{'='*60}")
    print(f"AutoGluon: {disease} (seed={seed}, mode={AUTOGLUON_MODE})")
    print(f"{'='*60}")

    if AUTOGLUON_MODE == "jumpstart":
        _, metrics = train_jumpstart(train, test, disease, disease_short)
    else:
        save_path = str(LOCAL_EXPORT_DIR / "autogluon" / disease)
        _, metrics = train_local(train, test, disease, save_path)

    results[disease] = metrics
    expected = BENCHMARK_F1[disease]
    status = "✓" if abs(metrics['f1'] - expected) < 0.05 else "⚠ DRIFT"
    print(f"  F1={metrics['f1']:.3f} (benchmark {expected:.3f}) {status}  Train={metrics['training_time_s']:.1f}s")

## 4. Verify Exported Models

In [ ]:
if AUTOGLUON_MODE == "local":
    print("Verifying exported AutoGluon models...")
    for disease in DISEASES:
        loaded = TabularPredictor.load(str(LOCAL_EXPORT_DIR / "autogluon" / disease), verbosity=0)
        train, test = get_split(combined, disease, OPTIMAL_SEEDS[disease])
        y_pred = loaded.predict(test.drop(columns=[disease]))
        f1 = round(f1_score(test[disease], y_pred, pos_label='1', zero_division=0), 4)
        match = '✓' if f1 == results[disease]['f1'] else '⚠'
        print(f"  {disease}: F1={f1} {match}")

## 5. Upload to S3

In [ ]:
if UPLOAD_TO_S3 and AUTOGLUON_MODE == "local":
    s3 = boto3.client('s3')
    for disease in DISEASES:
        local_dir = LOCAL_EXPORT_DIR / "autogluon" / disease
        s3_prefix = f"{S3_MODEL_PREFIX}/autogluon/{disease}"
        count = sum(1 for fp in local_dir.rglob("*") if fp.is_file()
                    and not (s3.upload_file(str(fp), S3_BUCKET, f"{s3_prefix}/{fp.relative_to(local_dir)}") or False))
        print(f"  {disease}: {count} files → s3://{S3_BUCKET}/{s3_prefix}/")
else:
    print(f"S3 upload skipped. Artifacts at: {LOCAL_EXPORT_DIR.resolve()}")

## Summary

| Disease | F1 | Benchmark | Status |
|---------|-----|-----------|--------|
| Hypertension | — | 0.909 | — |
| Stroke | — | 1.000 | — |
| CHD | — | 0.667 | — |
| Alzheimer's | — | 0.400 | — |

Artifacts saved to `models/exported/autogluon/{disease}/`
Load with: `TabularPredictor.load(path)`